Temporal Data Alignment





In [71]:
import os
import glob
import zipfile
from datetime import timedelta

import pandas as pd


BASE_DIR = "/content"

JISDOR_PATH = os.path.join(
    BASE_DIR,
    "jisdor_cleaned.xlsx"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "cleaned_aligned"
)

OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "news_jisdor_aligned.csv"
)

START_DATE = pd.Timestamp("2021-09-01")
END_DATE = pd.Timestamp("2026-09-01")


def find_processed_zip():
    # Find the processed P1 ZIP file in /content.

    zip_files = glob.glob(
        os.path.join(BASE_DIR, "*.zip")
    )

    if not zip_files:
        raise FileNotFoundError(
            "No ZIP file was found in /content."
        )

    processed_zips = [
        file for file in zip_files
        if "processed" in os.path.basename(file).lower()
    ]

    if not processed_zips:
        raise FileNotFoundError(
            "No processed ZIP file was found in /content."
        )

    zip_path = processed_zips[0]

    print(
        "Using processed ZIP:",
        os.path.basename(zip_path)
    )

    return zip_path


def extract_processed_data(zip_path):
    # Extract the processed P1 ZIP file.

    processed_folder = os.path.join(
        BASE_DIR,
        "processed"
    )

    if os.path.exists(processed_folder):
        print(
            "Processed folder already exists."
        )
        return

    print(
        "Extracting processed P1 data..."
    )

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as zip_ref:
        zip_ref.extractall(BASE_DIR)

    print(
        "Extraction completed."
    )


def find_news_file():
    # Find the combined P1 news dataset.

    expected_path = os.path.join(
        BASE_DIR,
        "processed",
        "articles_deduped_all_years.csv"
    )

    if os.path.exists(expected_path):
        return expected_path

    matches = glob.glob(
        os.path.join(
            BASE_DIR,
            "**",
            "articles_deduped_all_years.csv"
        ),
        recursive=True
    )

    if matches:
        return matches[0]

    raise FileNotFoundError(
        "articles_deduped_all_years.csv was not found."
    )


def load_news(news_path):

    #Load the processed news dataset and remove
    #duplicate records before temporal alignment.


    news = pd.read_csv(news_path)

    print("\nNews data loaded.")
    print(
        "Original rows:",
        len(news)
    )

    required_columns = {
        "GLOBALEVENTID",
        "date"
    }

    missing_columns = (
        required_columns -
        set(news.columns)
    )

    if missing_columns:
        raise ValueError(
            "Missing required columns: "
            + ", ".join(missing_columns)
        )

    original_rows = len(news)

    news = (
        news
        .drop_duplicates(
            keep="first"
        )
        .reset_index(drop=True)
    )

    exact_duplicates_removed = (
        original_rows -
        len(news)
    )

    print(
        "Exact duplicate rows removed:",
        exact_duplicates_removed
    )

    if news["GLOBALEVENTID"].duplicated().any():

        duplicate_id_count = (
            news["GLOBALEVENTID"]
            .duplicated()
            .sum()
        )

        print(
            "Duplicate article IDs found:",
            duplicate_id_count
        )

        if "n_sources_merged" in news.columns:

            news["n_sources_merged"] = pd.to_numeric(
                news["n_sources_merged"],
                errors="coerce"
            )

            news = (
                news
                .sort_values(
                    "n_sources_merged",
                    ascending=False,
                    na_position="last"
                )
                .drop_duplicates(
                    subset=["GLOBALEVENTID"],
                    keep="first"
                )
                .reset_index(drop=True)
            )

        else:

            news = (
                news
                .drop_duplicates(
                    subset=["GLOBALEVENTID"],
                    keep="first"
                )
                .reset_index(drop=True)
            )

    remaining_duplicates = (
        news["GLOBALEVENTID"]
        .duplicated()
        .sum()
    )

    print(
        "Duplicate article IDs after cleanup:",
        remaining_duplicates
    )

    print(
        "Rows after news cleanup:",
        len(news)
    )

    return news


def load_jisdor():


    if not os.path.exists(JISDOR_PATH):
        raise FileNotFoundError(
            f"JISDOR file not found: {JISDOR_PATH}"
        )

    jisdor = pd.read_excel(
        JISDOR_PATH,
        sheet_name="Informasi Kurs Jisdor",
        header=4,
        usecols=[0, 1, 2]
    )

    jisdor["date_clean"] = pd.to_datetime(
        jisdor["date_clean"],
        errors="coerce"
    )

    jisdor["usd_idr"] = pd.to_numeric(
        jisdor["usd_idr"],
        errors="coerce"
    )

    jisdor = (
        jisdor
        .dropna(
            subset=["date_clean"]
        )
        .copy()
    )

    jisdor = (
        jisdor
        .sort_values("date_clean")
        .drop_duplicates(
            subset=["date_clean"],
            keep="last"
        )
        .reset_index(drop=True)
    )

    print("\nJISDOR data loaded.")
    print(
        "Rows:",
        len(jisdor)
    )

    print(
        "First date:",
        jisdor["date_clean"].min().date()
    )

    print(
        "Last date:",
        jisdor["date_clean"].max().date()
    )

    print(
        "Missing USD/IDR:",
        jisdor["usd_idr"].isna().sum()
    )

    return jisdor


def create_jisdor_calendar(jisdor):

    # Creating a set of actual JISDOR observation dates.

    # Actual observations are used instead of assuming
    #every Monday-Friday is a trading day.


    jisdor_dates = set(
        jisdor["date_clean"]
        .dt.date
        .dropna()
    )

    if not jisdor_dates:
        raise ValueError(
            "No valid JISDOR observation dates found."
        )

    return jisdor_dates


def get_next_jisdor_date(
    start_date,
    jisdor_dates,
    last_jisdor_date
):

    # Find the same or next available JISDOR date.


    current_date = pd.Timestamp(
        start_date
    ).date()

    while current_date <= last_jisdor_date:

        if current_date in jisdor_dates:
            return current_date

        current_date += timedelta(
            days=1
        )

    return pd.NaT


def assign_target_date(
    news_date,
    jisdor_dates,
    last_jisdor_date
):

    # Assign the appropriate JISDOR observation date.

   # Same observation date:
        # use the same date.

   # Weekend or non-observation date:
      #  move forward to the next available JISDOR date.


    if pd.isna(news_date):
        return pd.NaT

    return get_next_jisdor_date(
        news_date,
        jisdor_dates,
        last_jisdor_date
    )


def prepare_news(news):

   # Convert news dates and restrict them to
    # the assignment period.


    news = news.copy()

    news["date"] = pd.to_datetime(
        news["date"],
        errors="coerce"
    )

    missing_dates = (
        news["date"]
        .isna()
        .sum()
    )

    print("\nNews date validation.")
    print(
        "Missing/invalid dates:",
        missing_dates
    )

    news = news.dropna(
        subset=["date"]
    ).copy()

    news = news[
        (news["date"] >= START_DATE) &
        (news["date"] <= END_DATE)
    ].copy()

    news["date"] = (
        news["date"]
        .dt.normalize()
    )

    news = (
        news
        .sort_values("date")
        .reset_index(drop=True)
    )

    print(
        "News rows in assignment period:",
        len(news)
    )

    print(
        "First news date:",
        news["date"].min().date()
    )

    print(
        "Last news date:",
        news["date"].max().date()
    )

    return news


def align_news_with_jisdor(
    news,
    jisdor
):

    # Align each news article with the same or
    # next available JISDOR observation date.


    news = prepare_news(
        news
    )

    jisdor_dates = create_jisdor_calendar(
        jisdor
    )

    last_jisdor_date = max(
        jisdor_dates
    )

    news["target_date"] = news["date"].apply(
        lambda x: assign_target_date(
            x,
            jisdor_dates,
            last_jisdor_date
        )
    )

    news["target_date"] = pd.to_datetime(
        news["target_date"],
        errors="coerce"
    )

    jisdor_merge = jisdor[
        [
            "date_clean",
            "usd_idr"
        ]
    ].copy()

    jisdor_merge = jisdor_merge.rename(
        columns={
            "date_clean": "target_date"
        }
    )

    jisdor_merge["target_date"] = pd.to_datetime(
        jisdor_merge["target_date"],
        errors="coerce"
    )

    aligned_data = news.merge(
        jisdor_merge,
        on="target_date",
        how="left"
    )

    return (
        aligned_data,
        jisdor_dates
    )


def validate_alignment(
    aligned_data,
    jisdor_dates
):
    # Validate the temporal alignment result.

    print("\nTemporal alignment validation.")

    missing_news_dates = (
        aligned_data["date"]
        .isna()
        .sum()
    )

    missing_target_dates = (
        aligned_data["target_date"]
        .isna()
        .sum()
    )

    missing_usd = (
        aligned_data["usd_idr"]
        .isna()
        .sum()
    )

    valid_target_dates = (
        aligned_data["target_date"]
        .dt.date
        .isin(jisdor_dates)
    )

    invalid_target_dates = (
        ~valid_target_dates
    ).sum()

    outside_period = (
        (aligned_data["date"] < START_DATE) |
        (aligned_data["date"] > END_DATE)
    ).sum()

    backward_mapping = (
        aligned_data["target_date"] <
        aligned_data["date"]
    ).sum()

    duplicate_ids = (
        aligned_data["GLOBALEVENTID"]
        .duplicated()
        .sum()
    )

    print(
        "Total aligned articles:",
        len(aligned_data)
    )

    print(
        "Missing news dates:",
        missing_news_dates
    )

    print(
        "Missing target dates:",
        missing_target_dates
    )

    print(
        "Missing USD/IDR values:",
        missing_usd
    )

    print(
        "Invalid target dates:",
        invalid_target_dates
    )

    print(
        "News outside assignment period:",
        outside_period
    )

    print(
        "Backward mappings:",
        backward_mapping
    )

    print(
        "Duplicate article IDs:",
        duplicate_ids
    )

    validation_passed = (
        missing_news_dates == 0
        and missing_target_dates == 0
        and missing_usd == 0
        and invalid_target_dates == 0
        and outside_period == 0
        and backward_mapping == 0
        and duplicate_ids == 0
    )

    if validation_passed:
        print(
            "\nValidation result: PASSED"
        )
    else:
        print(
            "\nValidation result: CHECK REQUIRED"
        )

    return validation_passed


def show_alignment_examples(aligned_data):
    # Show examples where dates were shifted.

    examples = aligned_data[
        [
            "date",
            "target_date",
            "usd_idr"
        ]
    ].drop_duplicates()

    shifted = examples[
        examples["date"] !=
        examples["target_date"]
    ]

    print("\nAlignment examples.")

    if len(shifted) > 0:

        print(
            shifted
            .head(10)
            .to_string(index=False)
        )

    else:

        print(
            "No shifted dates found."
        )


def generate_statistics(aligned_data):
    # Generate summary statistics.

    total_articles = len(
        aligned_data
    )

    same_day = (
        aligned_data["date"] ==
        aligned_data["target_date"]
    ).sum()

    shifted = (
        aligned_data["date"] !=
        aligned_data["target_date"]
    ).sum()

    unique_news_dates = (
        aligned_data["date"]
        .nunique()
    )

    unique_target_dates = (
        aligned_data["target_date"]
        .nunique()
    )

    print("\nAlignment statistics.")

    print(
        "Total articles:",
        total_articles
    )

    print(
        "Unique news dates:",
        unique_news_dates
    )

    print(
        "Unique JISDOR target dates:",
        unique_target_dates
    )

    print(
        "Same-day mappings:",
        same_day
    )

    print(
        "Shifted mappings:",
        shifted
    )

    if total_articles > 0:

        shifted_percentage = (
            shifted /
            total_articles *
            100
        )

        print(
            "Shifted percentage:",
            f"{shifted_percentage:.2f}%"
        )


def save_aligned_data(aligned_data):
    # Save the final aligned dataset.

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )

    aligned_data.to_csv(
        OUTPUT_PATH,
        index=False
    )

    print("\nFinal dataset saved to:")
    print(OUTPUT_PATH)

    print(
        "Rows:",
        len(aligned_data)
    )

    print(
        "Columns:",
        len(aligned_data.columns)
    )


def main():

    print(
        "NEWS + JISDOR TEMPORAL ALIGNMENT"
    )

    print(
        "\nAssignment period:",
        START_DATE.date(),
        "to",
        END_DATE.date()
    )

    zip_path = find_processed_zip()

    extract_processed_data(
        zip_path
    )

    news_path = find_news_file()

    print(
        "\nNews file:",
        news_path
    )

    news = load_news(
        news_path
    )

    jisdor = load_jisdor()

    aligned_data, jisdor_dates = (
        align_news_with_jisdor(
            news,
            jisdor
        )
    )

    validation_passed = (
        validate_alignment(
            aligned_data,
            jisdor_dates
        )
    )

    show_alignment_examples(
        aligned_data
    )

    generate_statistics(
        aligned_data
    )

    save_aligned_data(
        aligned_data
    )

    print("\nPipeline completed.")

    if validation_passed:
        print(
            "Temporal alignment completed successfully."
        )
    else:
        print(
            "Completed, but validation needs review."
        )

    return aligned_data


aligned_data = main()

NEWS + JISDOR TEMPORAL ALIGNMENT

Assignment period: 2021-09-01 to 2026-09-01
Using processed ZIP: processed-20260915T110708Z-1-001.zip
Processed folder already exists.

News file: /content/processed/articles_deduped_all_years.csv

News data loaded.
Original rows: 7163
Exact duplicate rows removed: 1129
Duplicate article IDs found: 6
Duplicate article IDs after cleanup: 0
Rows after news cleanup: 6028

JISDOR data loaded.
Rows: 1202
First date: 2021-09-01
Last date: 2026-09-01
Missing USD/IDR: 0

News date validation.
Missing/invalid dates: 0
News rows in assignment period: 5969
First news date: 2021-09-01
Last news date: 2026-09-01

Temporal alignment validation.
Total aligned articles: 5969
Missing news dates: 0
Missing target dates: 0
Missing USD/IDR values: 0
Invalid target dates: 0
News outside assignment period: 0
Backward mappings: 0
Duplicate article IDs: 0

Validation result: PASSED

Alignment examples.
      date target_date  usd_idr
2021-09-04  2021-09-06    14239
2021-09-05